**TEST: VICOCKTAIL DATASET - GREEDY BASELINE**

In [1]:
import os
import sys

PROJECT_DIR = os.path.dirname(os.getcwd())

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

In [2]:
import yaml
import torch

from srcs.datasets.vicocktail import Collator, load_vicocktail
from srcs.nets.e2e import get_model
from srcs.trainer.trainer import HFTrainer, build_metric_fn, preprocess_logits_for_metrics
from srcs.spm.text_transofm import TextTransform
from transformers import TrainingArguments

with open(os.path.join(PROJECT_DIR, "config.yaml"), "r", encoding="utf-8") as f:
    configs = yaml.safe_load(f)

TEST_SIZE = 1.0
BASELINE_CHECKPOINT = os.path.join(PROJECT_DIR,"checkpoints","baseline","checkpoint-305877")
SAVE_DIR = os.path.join(PROJECT_DIR, 'experiments', "results")
SAMPLE_COUNT = 5

os.makedirs(SAVE_DIR, exist_ok=True)

d:\projects\VietnameseVSR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = load_vicocktail(test_fraction=TEST_SIZE, splits=("test",))
text_transform = TextTransform()
test_collator = Collator(text_transform, "test")
ds

DatasetDict({
    test: Dataset({
        features: ['label', 'length', 'sample_id', 'video', 'video_length'],
        num_rows: 1167
    })
})

In [4]:
model = get_model(
        "baseline",
        text_transform.vocab_size,
        checkpoint_path=BASELINE_CHECKPOINT,
        **configs["model"],
    )

In [5]:
evaluation_args = TrainingArguments(
    output_dir=SAVE_DIR,
    label_names=["labels", "label_lengths"],
    per_device_eval_batch_size=configs["evaluation"]["batch_size"],
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    dataloader_num_workers=configs["evaluation"]["num_workers"],
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none",
)

In [6]:
trainer = HFTrainer(
    model=model,
    args=evaluation_args,
    eval_dataset=ds["test"],
    data_collator=test_collator,
    validation_collator=test_collator,
    compute_metrics=build_metric_fn(text_transform),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)
metrics = trainer.evaluate(metric_key_prefix="test")
trainer.log_metrics("greedy_CTC_baseline", metrics)
trainer.save_metrics("greedy_CTC_baseline", metrics)

Training Loss,Validation Loss,Step,Wer
No log,84.746811,0,0.615932


***** greedy_CTC_baseline metrics *****
  test_loss = 84.7468
  test_wer  =  0.6159


In [8]:
sample_items = [ds["test"][index] for index in range(SAMPLE_COUNT)]
sample_batch = test_collator(sample_items)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

with torch.inference_mode():
    predictions = model.decode(
        sample_batch["videos"].to(device),
        sample_batch["video_lengths"].to(device),
    )

references = [
    text_transform.decode(label[: int(length)])
    for label, length in zip(
        sample_batch["labels"], sample_batch["label_lengths"]
    )
]
hypotheses = [text_transform.decode(token_ids) for token_ids in predictions]

for index, (item, reference, hypothesis) in enumerate(
    zip(sample_items, references, hypotheses)
):
    print(f"Sample {index + 1}")
    print(f"Pred: {hypothesis}")
    print(f"Ref: {reference}")
    print()


Sample 1
Pred: khi mà mình có được những điều mà mình mới có thể cái một sống cái bỏ con rồi biết đứa ra đều đang đó làm kiếm kỹ làm để xác rất là đáng khó được không học ra
Ref: khi mà mình rõ được những cái điều mà mình ưu tiên trong cuộc sống thì mọi lựa chọn chúng mình đưa ra đều lấy nó làm kim chỉ nam để xác định là nên nói có hay là nó không học cách

Sample 2
Pred: tốt trong bài học là mình không trong ngày nay mình là cho không ra và trên đó
Ref: mười bài học mà mình học được trong năm nay mình đã chọn lọc ra và sẽ chia sẻ với bạn

Sample 3
Pred: sáng đọc sao về khoa học về việc tự học trong cuốn sách là khoa với thi năm vì không không bài gặp cái việc là
Ref: dơ sai ừn ợp seo lơn ninh khoa học về việc tự học trong cuốn sách này ngoài việc đề cập tới việc tự học thì nó cũng đề cập tới việc là

Sample 4
Pred: nhiều bạn tất cả về cái sự tập trung của con hẳn là nếu bạn sinh viên là cái việc mà bạn góp bị ở sinh viên tức là bạn đang bán kỹ năng của bạn có người án vì bạn hàng và t